# FOMNv Healpix maps: impact of the dust-footprint threshold E(B-V)

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-16
- Kernel: conda_py313_opsim53
- Context: SCOC footprint-shrinking study - variation of the dust-extinction threshold E(B-V) used to define the WFD footprint boundary in the v5.3.6 feature-scheduler simulations.
- This notebook: **01** of series `07_variateEVmV` - Healpix maps of the per-pixel number of visits (`NVisits`, the quantity underlying the official `fONv` Figure of Merit), for each dust-threshold footprint variant, and the pairwise difference maps between consecutive thresholds.
- Reference notebook for the metric definition: `../03_fbs5.3.6/Footprint.ipynb` (section "FP Comparison")
- Reference notebooks for the repository conventions (headers, `data_<TAG>/figs_<TAG>/` layout, dual PNG+PDF figure saving): `../06_MAF_DESC_TaskF/` series (e.g. `02_WL_DESC_TaskForce_demo.ipynb`)
- OpSim simulations analyzed (`/Users/dagoret/DATA/OpSim/`):
  - `shrink_fp_dust_0.050_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.080_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.120_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.150_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.199_v5.3.6_10yrs.db`
  - `baseline_v5.3.6_10yrs.db` (= `shrink_fp_dust_0.200_v5.3.6_10yrs.db`, stored under `sim_baseline/`)
  - `shrink_fp_dust_0.250_v5.3.6_10yrs.db`


## Notebook overview

**What "FOMNv" means here.** The official Figure of Merit `fONv` (SCOC "SRD" metric group) is a *scalar*: the number of visits reached at a reference sky area (825 sq deg by default), read off the cumulative curve "number of visits vs. sky area" built by sorting all Healpix pixels by their visit count. That cumulative curve - and therefore `fONv` - is entirely derived from one underlying **Healpix map**: the per-pixel number of visits, `NVisits`, computed with `rubin_sim.maf.metrics.CountMetric` on a `HealpixSlicer` (`nside=64`), with no SQL constraint (all visits, all filters, all 10 years). This is exactly the map produced as the `"..._fO_All_visits_HEAL"` bundle of `rubin_sim.maf.batches.fOBatch`, and is the same quantity plotted directly with `CountMetric` in `../03_fbs5.3.6/Footprint.ipynb`.

This notebook:
1. computes (or reuses cached results from `../03_fbs5.3.6/`) this `NVisits` Healpix map for the 7 dust-threshold variants of the v5.3.6 footprint;
2. plots all 7 maps side by side;
3. plots and saves the Healpix map of the **difference** in `NVisits` between each pair of *consecutive* dust thresholds - the six pairs requested: `0.080-0.050`, `0.120-0.080`, `0.150-0.120`, `0.199-0.150`, `baseline-0.199`, `0.250-baseline`;
4. summarizes the differences (mean/median/std/min/max) in a table, plus histograms;
5. cross-checks against the scalar `fONv` values in `summary.h5`, if available in this environment.


## 1. Imports

In [ ]:
import os
from os.path import join, isfile

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf

print("rubin_sim version:", rubin_sim.__version__)

## 2. Configuration

In [ ]:
NB_TAG = "FOMNV"
data_dir = f"data_01_{NB_TAG}"
figs_dir = f"figs_01_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

resultsDb = maf.db.ResultsDb(out_dir=data_dir)

In [ ]:
# Directory holding all the shrink_fp_dust_*.db OpSim databases
# (the baseline lives in a sim_baseline/ subfolder, as in the 06_MAF_DESC_TaskF notebooks)
OPSIM_DIR = "/Users/dagoret/DATA/OpSim"

# Cache directories to search for already-computed fO_All_visits_HEAL.npz bundles
# before re-running MAF on the full 10-year database (baseline, 0.080 and 0.120 are
# already cached from ../03_fbs5.3.6/Footprint.ipynb and v5.3.6_Update.ipynb).
CACHE_DIRS = [join("..", "03_fbs5.3.6")]

NSIDE = 64

# (run_name, dust threshold E(B-V)) sorted by increasing threshold
RUNS_INFO = [
    ("shrink_fp_dust_0.050_v5.3.6_10yrs", 0.050),
    ("shrink_fp_dust_0.080_v5.3.6_10yrs", 0.080),
    ("shrink_fp_dust_0.120_v5.3.6_10yrs", 0.120),
    ("shrink_fp_dust_0.150_v5.3.6_10yrs", 0.150),
    ("shrink_fp_dust_0.199_v5.3.6_10yrs", 0.199),
    ("baseline_v5.3.6_10yrs", 0.200),
    ("shrink_fp_dust_0.250_v5.3.6_10yrs", 0.250),
]
RUN_NAMES = [r for r, _ in RUNS_INFO]
DUST_THRESH = dict(RUNS_INFO)


def get_db_path(run_name):
    """Locate the OpSim sqlite database for a given run.

    Tries OPSIM_DIR directly, then OPSIM_DIR/sim_baseline (where the official
    baseline run is stored, per the 06_MAF_DESC_TaskF README convention).
    """
    fname = run_name + ".db"
    candidates = [join(OPSIM_DIR, fname), join(OPSIM_DIR, "sim_baseline", fname)]
    for c in candidates:
        if isfile(c):
            return c
    raise FileNotFoundError(f"OpSim db not found for run '{run_name}'. Tried: {candidates}")


for run_name, dust in RUNS_INFO:
    print(f"{run_name:40s} E(B-V) < {dust:.3f}  ->  {get_db_path(run_name)}")

## 3. NVisits Healpix map: compute (or reuse cached results) for each run

In [ ]:
def load_or_run_fo_nv(run_name, cache_dirs=CACHE_DIRS):
    """Return the fOBatch 'all visits' NVisits-per-pixel MetricBundle for one OpSim run.

    Reuses an already-computed .npz cache if found in `cache_dirs` or `data_dir` (avoids
    re-running MAF on the full 10-year database when a previous notebook already produced
    the exact same bundle), otherwise runs rubin_sim.maf.fOBatch on the OpSim database
    directly and caches the result in `data_dir`.
    """
    key = f"{run_name.replace('.', '_')}_fO_All_visits_HEAL"
    fname = f"{key}.npz"

    bundle_dict = maf.fOBatch(run_name=run_name)
    bundle = bundle_dict[key]

    for d in list(cache_dirs) + [data_dir]:
        candidate = join(d, fname)
        if isfile(candidate):
            bundle.read(candidate)
            print(f"[{run_name}] loaded cached map: {candidate}")
            return bundle

    dbpath = get_db_path(run_name)
    print(f"[{run_name}] no cache found -> running MAF on {dbpath}")
    g = maf.MetricBundleGroup(bundle_dict, dbpath, out_dir=data_dir, results_db=resultsDb, verbose=False)
    g.run_all()
    return bundle


nv_bundles = {}
for run_name, dust in RUNS_INFO:
    nv_bundles[run_name] = load_or_run_fo_nv(run_name)

npix = hp.nside2npix(NSIDE)
print(f"\nnside={NSIDE} -> npix={npix}, pixel area = {hp.nside2pixarea(NSIDE, degrees=True):.4f} deg^2")

## 4. Healpix skymaps of NVisits for all 7 footprint variants

In [ ]:
fig = plt.figure(figsize=(15, 12))
for i, (run_name, dust) in enumerate(RUNS_INFO, start=1):
    mval = nv_bundles[run_name].metric_values.filled(0)
    hp.mollview(
        mval,
        fig=fig.number,
        sub=(3, 3, i),
        min=0,
        max=1000,
        cmap="viridis",
        title=f"{run_name}\nE(B-V) < {dust:.3f}",
        unit="NVisits",
        cbar=True,
    )

fig.suptitle("FOMNv (per-pixel NVisits) - v5.3.6 dust-footprint variants", fontsize=16, y=1.02)
base = join(figs_dir, "FOMNv_healpix_allruns")
fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
fig.savefig(base + ".pdf", bbox_inches="tight")
print("Saved:", base + ".png/.pdf")
plt.show()

## 5. Difference maps between consecutive dust thresholds

For each pair of adjacent thresholds (sorted by increasing E(B-V) cut), the map plotted is
`NVisits(run_b) - NVisits(run_a)`, i.e. the *later* (larger-threshold, larger-footprint) run
minus the *earlier* one. A positive region means that loosening the dust cut from `run_a` to
`run_b` added visits there (typically newly-included low-Galactic-latitude sky); a negative
region means visits were redistributed away from it.


In [ ]:
# Consecutive-threshold pairs, in the order requested:
# 0.080-0.050, 0.120-0.080, 0.150-0.120, 0.199-0.150, baseline-0.199, 0.250-baseline
PAIRS = [(RUN_NAMES[i + 1], RUN_NAMES[i]) for i in range(len(RUN_NAMES) - 1)]
for run_b, run_a in PAIRS:
    print(f"{run_b}  -  {run_a}")

In [ ]:
def diff_map(run_b, run_a):
    """Per-pixel difference NVisits(run_b) - NVisits(run_a); masked pixels filled with 0."""
    map_b = nv_bundles[run_b].metric_values.filled(0)
    map_a = nv_bundles[run_a].metric_values.filled(0)
    return map_b - map_a


def plot_and_save_diff(run_b, run_a, vlim=None):
    dmap = diff_map(run_b, run_a)
    if vlim is None:
        vlim = max(np.percentile(np.abs(dmap), 99), 1.0)

    fig = plt.figure(figsize=(8, 5))
    hp.mollview(
        dmap,
        fig=fig.number,
        min=-vlim,
        max=vlim,
        cmap="RdBu_r",
        title=f"FOMNv difference: {run_b}\nminus {run_a}",
        unit=r"$\Delta$ NVisits",
    )
    hp.graticule()

    tag = f"{run_b}_MINUS_{run_a}".replace(".", "_")
    base = join(figs_dir, f"FOMNv_diff_{tag}")
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    plt.show()
    print("Saved:", base + ".png/.pdf", " (color scale +/-%.1f, 99th pct)" % vlim)
    return dmap

In [ ]:
diff_maps = {}
summary_rows = []
for run_b, run_a in PAIRS:
    print(f"=== {run_b}  -  {run_a} ===")
    dmap = plot_and_save_diff(run_b, run_a)
    diff_maps[(run_b, run_a)] = dmap
    summary_rows.append(
        {
            "pair": f"{run_b} - {run_a}",
            "dust_a": DUST_THRESH[run_a],
            "dust_b": DUST_THRESH[run_b],
            "mean": np.mean(dmap),
            "median": np.median(dmap),
            "std": np.std(dmap),
            "min": np.min(dmap),
            "max": np.max(dmap),
        }
    )

summary_df = pd.DataFrame(summary_rows).set_index("pair")
summary_df

## 6. Summary table and histograms of the differences

In [ ]:
summary_csv = join(data_dir, "FOMNv_diff_summary.csv")
summary_df.to_csv(summary_csv)
print("Saved:", summary_csv)
summary_df.round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (run_b, run_a) in zip(axes.flat, PAIRS):
    dmap = diff_maps[(run_b, run_a)]
    ax.hist(dmap, bins=100, color="steelblue")
    ax.axvline(0, color="k", linewidth=0.8)
    ax.set_title(f"{run_b}\n- {run_a}", fontsize=9)
    ax.set_xlabel(r"$\Delta$ NVisits")
    ax.set_ylabel("N pixels")
fig.tight_layout()

base = join(figs_dir, "FOMNv_diff_histograms")
fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
fig.savefig(base + ".pdf", bbox_inches="tight")
print("Saved:", base + ".png/.pdf")
plt.show()

## 7. Cross-check: scalar `fONv` Figure of Merit from `summary.h5` (if available)

In [ ]:
# The scalar fONv (number of visits at the 825 sq deg reference area) is normally read from the
# precomputed summary.h5 file (as in ../03_fbs5.3.6/Footprint.ipynb), rather than recomputed here.
# This cell is a best-effort cross-check and is skipped gracefully if summary.h5 is not reachable
# from this machine (e.g. when working purely from local OpSim .db files).
try:
    path_topsim = os.getenv("RUBIN_SIM_DATA_DIR")
    path_summary = join(path_topsim, "maf/fbs5.3.6/summary.h5")
    summaries = maf.get_metric_summaries(summary_source=path_summary)
    fonv_cols = [c for c in summaries.columns if "fONv" in c and "HealpixSlicer" in c]
    display(summaries.loc[RUN_NAMES, fonv_cols].round(1))
except Exception as exc:
    print("summary.h5 cross-check skipped (not available in this environment):", exc)

## Caveats

- Masked Healpix pixels (outside the survey footprint for a given run) are filled with 0 before
  differencing. Near the edge of the footprint this can make a difference map show a change from
  0 to N (or N to 0) rather than a genuine "extra visits" signal - this is expected and is exactly
  the dust-footprint boundary moving, which is the effect being studied here.
- All maps use the full 10-year, all-band, all-visit `NVisits` bundle (`constraint=None`), matching
  the `bundle` object (not `bundle2`, which used `night < 366`) in `../03_fbs5.3.6/Footprint.ipynb`.
- The colorbar limits for each difference map are set independently to the 99th percentile of
  `|difference|` for that pair, so the six panels are **not** on a common color scale. Compare
  absolute magnitudes via the summary table in Section 6, or fix a common `vlim` in
  `plot_and_save_diff(..., vlim=...)` if a shared scale is needed.


## References
- `../03_fbs5.3.6/Footprint.ipynb` - `CountMetric` / `HealpixSlicer` definition of `NVisits`, and the `fOBatch` FP-comparison section this notebook builds on.
- `../06_MAF_DESC_TaskF/` series - repository conventions (headers, `data_<TAG>/figs_<TAG>/`, dual PNG+PDF saving).
- `rubin_sim.maf.batches.fOBatch` source: https://github.com/lsst/rubin_sim/blob/main/rubin_sim/maf/batches/srd_batch.py
- `shrink_fp_dust_*.db` simulations: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/shrink_fp/
- Table of simulations: https://usdf-maf.slac.stanford.edu/
